# 05 — Visualization Prototyping & Storytelling: CO₂ & Greenhouse Gas Emissions

Prototype dashboard charts, map each visual to Data Visualization concepts, and produce storytelling insight cards. Every prototype is anchored to the OWID CO₂ dataset.

In [2]:
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)

PROJECT_ROOT  = Path("..").resolve()
RAW_PATH      = PROJECT_ROOT / "data" / "raw" / "owid-co2-data.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
AGG_DIR       = PROJECT_ROOT / "data" / "aggregated"
FIGURE_DIR    = PROJECT_ROOT / "reports" / "figures"
for p in [PROCESSED_DIR, AGG_DIR, FIGURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def section(t):
    print("\n" + "="*90); print(t); print("="*90)

section("Load datasets")

# Clustered snapshot (from NB04) — falls back to modern if NB04 not yet run
clustered_path = PROCESSED_DIR / "co2_clustered.csv"
modern_path    = PROCESSED_DIR / "co2_modern.csv"

if clustered_path.exists():
    df = pd.read_csv(clustered_path)
    print(f"Loaded clustered snapshot: {df.shape}")
elif modern_path.exists():
    df = pd.read_csv(modern_path)
    print(f"Loaded modern dataset (run NB04 for cluster labels): {df.shape}")
else:
    raise FileNotFoundError("Run notebooks 02 and 04 first to generate processed data.")

df_raw = pd.read_csv(RAW_PATH)
world  = df_raw[df_raw["country"] == "World"].copy()
modern = pd.read_csv(modern_path) if modern_path.exists() else df

# Latest snapshot per country
latest_year = int(modern["year"].max())
latest = modern[modern["year"] == latest_year].dropna(subset=["co2"]).copy()

print(f"Latest year: {latest_year} | Countries: {latest['country'].nunique()}")
display(df.head(3))



Load datasets
Loaded clustered snapshot: (164, 91)
Latest year: 2024 | Countries: 215


,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,co2_including_luc,co2_including_luc_growth_abs,co2_including_luc_growth_prct,co2_including_luc_per_capita,co2_including_luc_per_gdp,co2_including_luc_per_unit_energy,co2_per_capita,co2_per_gdp,co2_per_unit_energy,coal_co2,coal_co2_per_capita,consumption_co2,consumption_co2_per_capita,consumption_co2_per_gdp,cumulative_cement_co2,cumulative_co2,cumulative_co2_including_luc,cumulative_coal_co2,cumulative_flaring_co2,cumulative_gas_co2,cumulative_luc_co2,cumulative_oil_co2,cumulative_other_co2,energy_per_capita,energy_per_gdp,flaring_co2,flaring_co2_per_capita,gas_co2,gas_co2_per_capita,ghg_excluding_lucf_per_capita,ghg_per_capita,land_use_change_co2,land_use_change_co2_per_capita,methane,methane_per_capita,nitrous_oxide,nitrous_oxide_per_capita,oil_co2,oil_co2_per_capita,other_co2_per_capita,other_industry_co2,primary_energy_consumption,share_global_cement_co2,share_global_co2,share_global_co2_including_luc,share_global_coal_co2,share_global_cumulative_cement_co2,share_global_cumulative_co2,share_global_cumulative_co2_including_luc,share_global_cumulative_coal_co2,share_global_cumulative_flaring_co2,share_global_cumulative_gas_co2,share_global_cumulative_luc_co2,share_global_cumulative_oil_co2,share_global_cumulative_other_co2,share_global_flaring_co2,share_global_gas_co2,share_global_luc_co2,share_global_oil_co2,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share,co2_per_energy,fossil_share,luc_share,gdp_per_capita,co2_growth_cat,income_tier,dominant_fuel,decade,Cluster,Cluster_Name,PCA_1,PCA_2
0,Afghanistan,2024,AFG,42647502.0,5.330347e+10,0.010624,0.000249,10.825998,0.309679,2.944744,29.073744,2.883896,11.011505,0.681722,0.404172,736.928955,0.253848,0.190792,274.405365,3.894407,0.091316,NaN,NaN,NaN,2.552634,242.635437,611.707764,62.914925,5.956043,20.494799,862.501587,150.717041,NaN,925.085266,0.846218,0.000000,0.000000,0.126113,0.002957,0.371494,0.935249,18.247746,0.427874,16.617899,0.389657,4.346074,0.101907,6.794853,0.159326,NaN,NaN,39.452576,0.000721,0.028048,0.067325,0.024640,0.005137,0.013122,0.022232,0.007403,0.029571,0.007410,0.095099,0.023623,NaN,0.000000,0.001574,0.397944,0.054487,NaN,0.080222,0.000585,0.000620,0.001346,0.000141,39.886021,15.843285,NaN,NaN,0.274405,0.999019,0.627637,1313.577777,Moderate growth,Unknown,LUC,2020s,1,Rapidly industrialising,-1.194005,-0.336660
1,Albania,2024,ALB,2791756.0,3.617101e+10,0.906674,0.324768,4.444448,0.027115,0.613832,4.024407,-0.087898,-2.137440,1.441532,0.119427,157.242676,1.591990,0.124362,173.654633,0.495061,0.177330,5.382012,1.914176,0.157208,27.422752,307.938110,548.291687,70.195389,0.152939,16.644436,393.309235,193.522583,NaN,9167.564453,0.715189,0.000658,0.000236,0.085966,0.030793,1.983947,2.402803,-0.420041,-0.150458,2.173883,0.778679,0.710292,0.254425,2.956089,1.058864,NaN,NaN,25.593603,0.061561,0.011515,0.009319,0.003132,0.055183,0.016653,0.019927,0.008260,0.000759,0.006018,0.043366,0.030332,NaN,0.000158,0.001073,-0.009160,0.023704,NaN,0.025627,0.000132,0.000258,0.000430,0.000040,6.708038,5.538696,0.964679,21.838495,0.173655,0.795850,-0.104373,12792.059169,Moderate growth,Unknown,Oil,2020s,1,Rapidly industrialising,-1.207716,-0.123481
2,Algeria,2024,DZA,46814302.0,5.958201e+11,12.836498,0.274200,198.203186,-4.643472,-2.289152,200.742371,-4.781792,-2.326632,4.288056,0.328449,264.088623,4.233817,0.323552,260.748169,0.605761,0.012940,NaN,NaN,NaN,253.360001,5476.567871,6544.149414,119.981392,781.888000,2469.692139,1307.129395,1851.646606,NaN,16237.188477,1.232331,14.818216,0.316532,101.799698,2.174543,5.605033,5.898497,2.539188,0.054240,69.212425,1.478446,8.114459,0.173333,68.143013,1.455603,NaN,NaN,760.132629,0.871561,0.513499,0.464853,0.003833,0.509841,0.296171,0.237839,0.014118,3.8819

In [3]:
section("Dashboard concept map")

concept_map = pd.DataFrame([
    ["Executive Overview",
     "Annotated KPI table + Global trend line",
     "How much CO₂ has the world emitted and how has it changed?",
     "Trend line, annotation, preattentive highlighting, reference lines"],
    ["Emissions Inequality",
     "Diverging bar — per-capita top & bottom",
     "How unequal is responsibility for CO₂ across nations?",
     "Sorted bar, position encoding, color to encode category"],
    ["Fuel Mix Transition",
     "Stacked area chart — world fuel sources",
     "How has the energy mix shifted from coal to gas and renewables?",
     "Area chart, proportional stacking, time series, color legend"],
    ["Wealth vs Emissions",
     "Bubble scatter — GDP per cap vs CO₂ per cap",
     "Does wealth always mean more emissions? Is there a Kuznets curve?",
     "Scatter plot, bubble size, log scale, trendline, color segmentation"],
    ["Country Archetypes",
     "PCA map + Cluster radar",
     "What distinct country emissions profiles exist?",
     "PCA, clustering, radar chart, Gestalt enclosure"],
    ["Temperature Responsibility",
     "Horizontal bar — temperature change attribution",
     "Which countries have contributed most to global warming?",
     "Length encoding, sorted bar, cumulative framing"],
    ["Storytelling",
     "Insight cards",
     "What are the 6 key takeaways for a climate-aware audience?",
     "Narrative flow, progressive disclosure, visual hierarchy"],
], columns=["Dashboard Section", "Chart Type", "Main Question", "DataViz Concepts"])
display(concept_map)
concept_map.to_csv(PROCESSED_DIR / "dashboard_concept_map.csv", index=False)
print("Saved dashboard_concept_map.csv")



Dashboard concept map


,Dashboard Section,Chart Type,Main Question,DataViz Concepts
0,Executive Overview,Annotated KPI table + Global trend line,How much CO₂ has the world emitted and how has...,"Trend line, annotation, preattentive highlight..."
1,Emissions Inequality,Diverging bar — per-capita top & bottom,How unequal is responsibility for CO₂ across n...,"Sorted bar, position encoding, color to encode..."
2,Fuel Mix Transition,Stacked area chart — world fuel sources,How has the energy mix shifted from coal to ga...,"Area chart, proportional stacking, time series..."
3,Wealth vs Emissions,Bubble scatter — GDP per cap vs CO₂ per cap,Does wealth always mean more emissions? Is the...,"Scatter plot, bubble size, log scale, trendlin..."
4,Country Archetypes,PCA map + Cluster radar,What distinct country emissions profiles exist?,"PCA, clustering, radar chart, Gestalt enclosure"
5,Temperature Responsibility,Horizontal bar — temperature change attribution,Which countries have contributed most to globa...,"Length encoding, sorted bar, cumulative framing"
6,Storytelling,Insight cards,What are the 6 key takeaways for a climate-awa...,"Narrative flow, progressive disclosure, visual..."


Saved dashboard_concept_map.csv


In [4]:
section("Prototype 1 — Executive overview: global CO₂ trend with annotated milestones")

latest_world = world[world["year"] == world["year"].max()].iloc[0]
base_1990    = world[world["year"] == 1990]["co2"].values[0]
base_2000    = world[world["year"] == 2000]["co2"].values[0]

kpis = pd.DataFrame({
    "KPI": [
        "Latest data year",
        "World total CO₂",
        "Change vs 1990",
        "Change vs 2000",
        "Top emitter",
        "Lowest per-capita emitter (pop > 5M)",
        "Countries tracked",
    ],
    "Value": [
        str(int(latest_world["year"])),
        f"{latest_world['co2']:,.0f} Mt",
        f"{(latest_world['co2']/base_1990 - 1)*100:+.1f}%",
        f"{(latest_world['co2']/base_2000 - 1)*100:+.1f}%",
        latest.nlargest(1,"co2")["country"].values[0],
        latest[latest["population"] > 5_000_000].nsmallest(1,"co2_per_capita")["country"].values[0],
        f"{latest['country'].nunique():,}",
    ]
})
display(kpis)

# Annotated global trend
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=world["year"], y=world["co2"],
    mode="lines", name="World CO₂ (Mt)",
    fill="tozeroy", fillcolor="rgba(230,57,70,0.12)",
    line=dict(color="#e63946", width=2.5)
))

milestones = [
    (1950, "Post-war boom",  world[world["year"]==1950]["co2"].values[0]),
    (1973, "Oil crisis",     world[world["year"]==1973]["co2"].values[0]),
    (1990, "Rio Summit",     world[world["year"]==1990]["co2"].values[0]),
    (2008, "Financial crash",world[world["year"]==2008]["co2"].values[0]),
    (2015, "Paris Agreement",world[world["year"]==2015]["co2"].values[0]),
    (2020, "COVID-19",       world[world["year"]==2020]["co2"].values[0]),
]
for yr, label, yval in milestones:
    fig.add_vline(x=yr, line_dash="dot", line_color="gray", opacity=0.6)
    fig.add_annotation(x=yr, y=yval * 1.08, text=label, showarrow=False,
                       font=dict(size=9, color="#444"), textangle=-40)

fig.update_layout(
    title="Global CO₂ Emissions 1750–Present — Key Milestones",
    xaxis_title="Year", yaxis_title="Million tonnes CO₂",
    height=520, template="plotly_white"
)
fig.show()
print("DataViz concepts: time series, area fill, annotation, preattentive reference lines")



Prototype 1 — Executive overview: global CO₂ trend with annotated milestones


,KPI,Value
0,Latest data year,2024
1,World total CO₂,"38,599 Mt"
2,Change vs 1990,+69.8%
3,Change vs 2000,+51.3%
4,Top emitter,China
5,Lowest per-capita emitter (pop > 5M),Democratic Republic of Congo
6,Countries tracked,215


DataViz concepts: time series, area fill, annotation, preattentive reference lines


In [5]:
section("Prototype 2 — Emissions inequality: per-capita diverging bar")

big = latest[latest["population"] > 2_000_000].dropna(subset=["co2_per_capita"]).copy()
world_avg = world[world["year"] == latest_year]["co2_per_capita"].values[0]
big["vs_avg"] = big["co2_per_capita"] - world_avg
top15 = big.nlargest(15, "co2_per_capita").copy()
bot15 = big.nsmallest(15, "co2_per_capita").copy()

combo = pd.concat([top15, bot15]).drop_duplicates("country").sort_values("co2_per_capita")

fig = px.bar(
    combo, x="co2_per_capita", y="country", orientation="h",
    color="vs_avg",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    title=f"CO₂ per Capita vs World Average ({latest_year}) — Top & Bottom 15<br>"
          f"<sub>World avg = {world_avg:.1f} t/person. Red = above average. Blue = below.</sub>",
    labels={"co2_per_capita": "t CO₂ per person", "country": "", "vs_avg": "Δ from world avg"}
)
fig.add_vline(x=world_avg, line_dash="dash", line_color="black",
              annotation_text=f"World avg {world_avg:.1f}t", annotation_position="top right")
fig.update_layout(height=680, template="plotly_white")
fig.show()
print("DataViz concepts: sorted bar, diverging color scale, reference line annotation, position encoding")



Prototype 2 — Emissions inequality: per-capita diverging bar


DataViz concepts: sorted bar, diverging color scale, reference line annotation, position encoding


In [6]:
section("Prototype 3 — Fuel mix transition: stacked area chart")

fuel_cols   = ["coal_co2","oil_co2","gas_co2","cement_co2","flaring_co2","land_use_change_co2"]
fuel_labels = {"coal_co2":"Coal","oil_co2":"Oil","gas_co2":"Gas",
               "cement_co2":"Cement","flaring_co2":"Flaring","land_use_change_co2":"Land use change"}
fuel_colors = {
    "Coal":"#3d405b","Oil":"#e07a5f","Gas":"#f2cc8f",
    "Cement":"#81b29a","Flaring":"#c77dff","Land use change":"#70c1b3"
}

fuel_cols = [c for c in fuel_cols if c in world.columns]
world_fuel = world[world["year"] >= 1900][["year"] + fuel_cols].dropna(subset=["coal_co2","oil_co2","gas_co2"]).copy()
world_fuel.rename(columns=fuel_labels, inplace=True)
fuel_label_cols = [fuel_labels[c] for c in fuel_cols if c in fuel_labels]

world_long = world_fuel.melt(id_vars="year", value_vars=fuel_label_cols, var_name="Fuel", value_name="CO₂ (Mt)")

fig = px.area(
    world_long, x="year", y="CO₂ (Mt)", color="Fuel",
    color_discrete_map=fuel_colors,
    title="Global CO₂ Emissions by Fuel Source (1900–present) — Stacked Area",
    labels={"year": "Year"}
)
fig.add_vline(x=1990, line_dash="dot", line_color="gray",
              annotation_text="Rio Earth Summit (1992)", annotation_position="top left")
fig.update_layout(height=520, template="plotly_white")
fig.show()

# Normalised 100% stacked
total = world_fuel[fuel_label_cols].sum(axis=1).replace(0, np.nan)
norm_fuel = world_fuel.copy()
for c in fuel_label_cols:
    norm_fuel[c] = norm_fuel[c] / total * 100
norm_long = norm_fuel.melt(id_vars="year", value_vars=fuel_label_cols, var_name="Fuel", value_name="Share (%)")

fig2 = px.area(
    norm_long, x="year", y="Share (%)", color="Fuel",
    color_discrete_map=fuel_colors,
    title="Fuel Mix Share (%) Over Time — Proportional Stacked Area",
    labels={"year":"Year"}
)
fig2.update_layout(height=460, template="plotly_white", yaxis=dict(ticksuffix="%"))
fig2.show()
print("DataViz concepts: stacked area, proportional area, time series, color legend, annotation")



Prototype 3 — Fuel mix transition: stacked area chart


DataViz concepts: stacked area, proportional area, time series, color legend, annotation


In [7]:
section("Prototype 4 — Wealth vs Emissions: bubble scatter (Environmental Kuznets Curve)")

econ = modern.dropna(subset=["gdp_per_capita","co2_per_capita","population"]).copy()
econ_snap = econ.sort_values("year").groupby("country").last().reset_index()
econ_snap = econ_snap[econ_snap["population"] > 1_000_000]

# Size column — cap to avoid giant bubbles
econ_snap["pop_scaled"] = np.sqrt(econ_snap["population"] / 1e6).clip(0.5, 40)

fig = px.scatter(
    econ_snap, x="gdp_per_capita", y="co2_per_capita",
    color="income_tier" if "income_tier" in econ_snap.columns else "co2_per_capita",
    size="pop_scaled",
    hover_name="country",
    log_x=True,
    trendline="lowess",
    title=f"GDP per Capita vs CO₂ per Capita — Bubble = Population ({latest_year})<br>"
          "<sub>Log x-axis. LOWESS trendline. Environmental Kuznets Curve exploration.</sub>",
    labels={"gdp_per_capita": "GDP per capita (USD, log scale)", "co2_per_capita": "CO₂ per capita (t)"}
)
fig.update_layout(height=600, template="plotly_white")
fig.show()

# Highlight: high income countries declining?
if "income_tier" in econ_snap.columns:
    high = modern[modern.get("income_tier", pd.Series()) == "High income"] if "income_tier" in modern.columns else pd.DataFrame()
    if not high.empty:
        trend_high = high.dropna(subset=["co2_per_capita"]).groupby("year")["co2_per_capita"].mean().reset_index()
        trend_low  = modern[modern.get("income_tier", pd.Series()) == "Low income"].dropna(subset=["co2_per_capita"]).groupby("year")["co2_per_capita"].mean().reset_index() if "income_tier" in modern.columns else pd.DataFrame()
        fig2 = go.Figure()
        fig2.add_trace(go.Scatter(x=trend_high["year"], y=trend_high["co2_per_capita"],
                                  mode="lines", name="High income", line=dict(color="#e63946")))
        if not trend_low.empty:
            fig2.add_trace(go.Scatter(x=trend_low["year"], y=trend_low["co2_per_capita"],
                                      mode="lines", name="Low income", line=dict(color="#457b9d")))
        fig2.update_layout(title="CO₂ per Capita Trend by Income Group (1990+)",
                           xaxis_title="Year", yaxis_title="t CO₂ per person",
                           height=440, template="plotly_white")
        fig2.show()
print("DataViz concepts: bubble scatter, log scale, LOWESS trendline, size encoding, colour segmentation")



Prototype 4 — Wealth vs Emissions: bubble scatter (Environmental Kuznets Curve)


DataViz concepts: bubble scatter, log scale, LOWESS trendline, size encoding, colour segmentation


In [8]:
section("Prototype 5 — Country archetypes: PCA map + cluster profile radar")

if "Cluster_Name" in df.columns and "PCA_1" in df.columns:
    fig = px.scatter(
        df.dropna(subset=["PCA_1","PCA_2"]),
        x="PCA_1", y="PCA_2",
        color="Cluster_Name",
        hover_name="country",
        opacity=0.8,
        hover_data=["co2_per_capita","gdp_per_capita","fossil_share"],
        title="PCA Country Map — Emission Profile Space by Cluster",
        labels={"PCA_1": "PC1 (energy intensity)", "PCA_2": "PC2 (economic development)"}
    )
    fig.update_layout(height=620, template="plotly_white")
    fig.show()

    # Cluster size bar
    counts = df["Cluster_Name"].value_counts().reset_index()
    counts.columns = ["Cluster_Name", "Countries"]
    counts["Pct"] = (counts["Countries"] / len(df) * 100).round(1)
    fig2 = px.bar(
        counts.sort_values("Countries"), x="Countries", y="Cluster_Name",
        orientation="h",
        text=counts.sort_values("Countries")["Pct"].astype(str) + "%",
        title="Country Cluster Size",
        labels={"Countries": "# Countries", "Cluster_Name": ""}
    )
    fig2.update_traces(textposition="outside")
    fig2.update_layout(height=360, template="plotly_white")
    fig2.show()
else:
    print("Cluster / PCA columns not found — run notebook 04 first.")
    # Fallback: income tier breakdown
    inc = latest["income_tier"].value_counts().reset_index() if "income_tier" in latest.columns else pd.DataFrame()
    if not inc.empty:
        inc.columns = ["income_tier","Countries"]
        fig = px.bar(inc.sort_values("Countries"), x="Countries", y="income_tier", orientation="h",
                     title=f"Countries by Income Tier ({latest_year})")
        fig.update_layout(height=360)
        fig.show()
print("DataViz concepts: PCA, clustering, scatter plot, Gestalt enclosure, colour encoding")



Prototype 5 — Country archetypes: PCA map + cluster profile radar


DataViz concepts: PCA, clustering, scatter plot, Gestalt enclosure, colour encoding


In [9]:
section("Prototype 6 — Temperature responsibility: cumulative attribution bar")

temp_col = "temperature_change_from_co2"
temp_ghg = "temperature_change_from_ghg"

if temp_col in latest.columns:
    top_temp = (
        latest
        .dropna(subset=[temp_col])
        .nlargest(20, temp_col)
        [["country", temp_col, temp_ghg if temp_ghg in latest.columns else temp_col, "cumulative_co2"]]
        .sort_values(temp_col, ascending=True)
        .reset_index(drop=True)
    )
    display(top_temp.round(4))

    fig = go.Figure()
    fig.add_trace(go.Bar(
        y=top_temp["country"],
        x=top_temp[temp_col],
        orientation="h",
        name="From CO₂",
        marker_color="#e63946"
    ))
    if temp_ghg in top_temp.columns:
        fig.add_trace(go.Bar(
            y=top_temp["country"],
            x=top_temp[temp_ghg] - top_temp[temp_col],
            orientation="h",
            name="Additional from GHGs",
            marker_color="#f4a261"
        ))
    fig.update_layout(
        barmode="stack",
        title="Top 20 Countries by Estimated Temperature Rise Contribution (°C)<br>"
              "<sub>Historical cumulative attribution based on OWID methodology</sub>",
        xaxis_title="°C warming attributed",
        height=600,
        template="plotly_white"
    )
    fig.show()
    print("DataViz concepts: stacked horizontal bar, sorted encoding, cumulative framing, dual-metric bars")
else:
    print("temperature_change_from_co2 column not available in latest snapshot.")



Prototype 6 — Temperature responsibility: cumulative attribution bar


,country,temperature_change_from_co2,temperature_change_from_ghg,cumulative_co2
0,Colombia,0.0108,0.0151,3802.0474
1,Kazakhstan,0.0108,0.0138,14937.4404
2,Democratic Republic of Congo,0.0110,0.0133,238.6997
3,Italy,0.0118,0.0148,26005.9434
4,South Africa,0.0120,0.0167,22826.9668
5,Poland,0.0140,0.0178,29043.9180
6,Mexico,0.0147,0.0220,22015.8516
7,Australia,0.0148,0.0254,20039.6699
8,France,0.0170,0.0218,40047.5664
9,Ukraine,0.0179,0.0228,31236.4961


DataViz concepts: stacked horizontal bar, sorted encoding, cumulative framing, dual-metric bars


In [10]:
section("Prototype 7 — Slopegraph: CO₂ per capita change 2000 → latest year for top emitters")

top_countries = latest.nlargest(15, "co2")["country"].tolist()
slope_data = modern[modern["country"].isin(top_countries) & modern["year"].isin([2000, latest_year])].copy()
slope_data = slope_data.dropna(subset=["co2_per_capita"])

fig = go.Figure()
colors = px.colors.qualitative.Set2
for i, country in enumerate(top_countries):
    sub = slope_data[slope_data["country"] == country].sort_values("year")
    if len(sub) < 2:
        continue
    c = colors[i % len(colors)]
    fig.add_trace(go.Scatter(
        x=sub["year"], y=sub["co2_per_capita"],
        mode="lines+markers+text",
        name=country,
        line=dict(color=c, width=2),
        marker=dict(size=8, color=c),
        text=[None, country],
        textposition="middle right",
        textfont=dict(size=9)
    ))

fig.update_layout(
    title=f"Slopegraph — CO₂ per Capita Change: 2000 → {latest_year} (Top 15 Total Emitters)",
    xaxis=dict(tickvals=[2000, latest_year], ticktext=["2000", str(latest_year)]),
    yaxis_title="t CO₂ per capita",
    showlegend=False,
    height=560,
    template="plotly_white"
)
fig.show()
print("DataViz concepts: slopegraph, before/after comparison, annotation, parallel coordinates by time")



Prototype 7 — Slopegraph: CO₂ per capita change 2000 → latest year for top emitters


DataViz concepts: slopegraph, before/after comparison, annotation, parallel coordinates by time


In [11]:
section("Prototype 8 — Choropleth: CO₂ per capita world map")

map_data = latest.dropna(subset=["iso_code","co2_per_capita"]).copy()

fig = px.choropleth(
    map_data,
    locations="iso_code",
    color="co2_per_capita",
    hover_name="country",
    hover_data={"co2": True, "share_global_co2": True, "income_tier": True},
    color_continuous_scale="YlOrRd",
    title=f"CO₂ per Capita by Country ({latest_year})",
    labels={"co2_per_capita": "t CO₂ per person"}
)
fig.update_layout(height=520, geo=dict(showframe=False, showcoastlines=True))
fig.show()

# Bubble map: total CO₂
fig2 = px.scatter_geo(
    map_data.dropna(subset=["co2"]),
    locations="iso_code",
    size="co2",
    color="income_tier" if "income_tier" in map_data.columns else "co2",
    hover_name="country",
    projection="natural earth",
    title=f"Total CO₂ Emissions by Country ({latest_year}) — Bubble = magnitude",
    labels={"co2":"Total CO₂ (Mt)"}
)
fig2.update_layout(height=520)
fig2.show()
print("DataViz concepts: choropleth, bubble map, geographic encoding, spatial pattern recognition")



Prototype 8 — Choropleth: CO₂ per capita world map


DataViz concepts: choropleth, bubble map, geographic encoding, spatial pattern recognition


In [12]:
section("Storytelling cards — key insights for a climate-aware audience")

# Pull live stats from data
top3 = latest.nlargest(3,"co2")
top3_share = top3["share_global_co2"].sum() if "share_global_co2" in top3.columns else None
high_cap = latest[latest["population"]>2e6].nlargest(1,"co2_per_capita")["country"].values[0]
low_cap  = latest[latest["population"]>5e6].nsmallest(1,"co2_per_capita")["country"].values[0]
high_cap_val = round(latest[latest["country"]==high_cap]["co2_per_capita"].values[0],1)
low_cap_val  = round(latest[latest["country"]==low_cap]["co2_per_capita"].values[0],1)
ratio = round(high_cap_val / max(low_cap_val, 0.01), 0)
chg_1990 = round((latest_world["co2"] / world[world["year"]==1990]["co2"].values[0] - 1)*100, 1)

cards = pd.DataFrame([
    [1,
     f"Global CO₂ is {chg_1990:+.1f}% vs 1990 — and still rising",
     f"World emissions rose sharply since industrialisation, pausing briefly during recessions and COVID-19.",
     "Annotated global trend line",
     "Time series, annotation, reference lines, preattentive attributes"],
    [2,
     f"3 countries emit half the world's CO₂",
     f"{top3['country'].tolist()} collectively produce ~{top3_share:.0f}% of global CO₂." if top3_share else "China, USA, and India alone produce ~50% of global CO₂.",
     "Sorted bar + share label",
     "Length encoding, part-to-whole framing"],
    [3,
     f"Per-capita inequality spans {ratio:.0f}×",
     f"{high_cap} emits {high_cap_val} t/person vs {low_cap} at {low_cap_val} t/person — a {ratio:.0f}× gap.",
     "Diverging per-capita bar",
     "Diverging colour scale, sorted bar, reference annotation"],
    [4,
     "Coal drove past growth; gas and oil shape today",
     "Coal dominated the 20th century. Today, oil and gas dominate the mix in rich nations while coal drives Asia's growth.",
     "Stacked area fuel mix chart",
     "Stacked area, proportional encoding, time series, colour legend"],
    [5,
     "Wealth and emissions are linked — but the link is weakening",
     "High-income nations are decarbonising. Some middle-income nations are rising rapidly. The Kuznets curve is real but slow.",
     "Bubble scatter GDP vs CO₂",
     "Bubble scatter, log scale, LOWESS trendline, colour segmentation"],
    [6,
     "Historical responsibility differs from current emissions",
     "The US and EU have contributed most to cumulative warming, even though China now leads annual emissions.",
     "Temperature attribution stacked bar + choropleth",
     "Stacked bar, cumulative framing, geographic encoding"],
], columns=["Order","Headline","Evidence","Recommended Visual","DataViz Concepts"])
display(cards)

cards.to_csv(PROCESSED_DIR / "storytelling_cards.csv", index=False)
print("Saved storytelling_cards.csv")



Storytelling cards — key insights for a climate-aware audience


,Order,Headline,Evidence,Recommended Visual,DataViz Concepts
0,1,Global CO₂ is +69.8% vs 1990 — and still rising,World emissions rose sharply since industriali...,Annotated global trend line,"Time series, annotation, reference lines, prea..."
1,2,3 countries emit half the world's CO₂,"['China', 'United States', 'India'] collective...",Sorted bar + share label,"Length encoding, part-to-whole framing"
2,3,Per-capita inequality spans 413×,Qatar emits 41.3 t/person vs Democratic Republ...,Diverging per-capita bar,"Diverging colour scale, sorted bar, reference ..."
3,4,Coal drove past growth; gas and oil shape today,"Coal dominated the 20th century. Today, oil an...",Stacked area fuel mix chart,"Stacked area, proportional encoding, time seri..."
4,5,Wealth and emissions are linked — but the link...,High-income nations are decarbonising. Some mi...,Bubble scatter GDP vs CO₂,"Bubble scatter, log scale, LOWESS trendline, c..."
5,6,Historical responsibility differs from current...,The US and EU have contributed most to cumulat...,Temperature attribution stacked bar + choropleth,"Stacked bar, cumulative framing, geographic en..."


Saved storytelling_cards.csv


In [13]:
section("Final export summary")

print("Outputs written by this notebook:")
for f in sorted(PROCESSED_DIR.iterdir()):
    if f.suffix == ".csv":
        print(f"  {f.name}")
print()
print("Visual prototypes produced (8 charts):")
prototypes = [
    "1. Annotated global CO₂ trend line (Executive Overview)",
    "2. Diverging per-capita bar — inequality (Emissions Inequality)",
    "3. Stacked area — fuel mix 1900+ (Fuel Mix Transition)",
    "4. Bubble scatter — GDP vs CO₂ / Kuznets (Wealth vs Emissions)",
    "5. PCA country map + cluster size bar (Country Archetypes)",
    "6. Temperature attribution stacked bar (Historical Responsibility)",
    "7. Slopegraph — 2000 → present top emitters (Before/After)",
    "8. Choropleth + bubble map (Geographic View)",
]
for p in prototypes:
    print(f"  {p}")
print()
print("6 storytelling cards with headlines, evidence, and recommended chart type saved.")



Final export summary
Outputs written by this notebook:
  cluster_profiles.csv
  cluster_sizes.csv
  co2_audit_preview.csv
  co2_clustered.csv
  co2_features.csv
  co2_modern.csv
  dashboard_concept_map.csv
  eda_insights.csv
  eda_key_insights.csv
  smartphone_audit_preview.csv
  smartphone_clustered.csv
  smartphone_features.csv
  smartphone_powerbi_model.csv
  storytelling_cards.csv

Visual prototypes produced (8 charts):
  1. Annotated global CO₂ trend line (Executive Overview)
  2. Diverging per-capita bar — inequality (Emissions Inequality)
  3. Stacked area — fuel mix 1900+ (Fuel Mix Transition)
  4. Bubble scatter — GDP vs CO₂ / Kuznets (Wealth vs Emissions)
  5. PCA country map + cluster size bar (Country Archetypes)
  6. Temperature attribution stacked bar (Historical Responsibility)
  7. Slopegraph — 2000 → present top emitters (Before/After)
  8. Choropleth + bubble map (Geographic View)

6 storytelling cards with headlines, evidence, and recommended chart type saved.
